# 08. Parameter and FLOPs Analysis

## 学習目標
- パラメータ構成比と、学習 FLOPs の2つの見積り（厳密 vs 6ND 近似）の差を理解する
- 「パラメータ数 ≠ 計算量」を数値で確認する

## 数式
- 順伝播 FLOPs/token（context T, MAC=2FLOP）は各層で
  $\text{qkv}=6d^2,\ \text{attn}\approx 4Td,\ \text{proj}=2d^2,\ \text{MLP}\approx 16d^2$、
  加えて lm_head $=2dV$。
- 学習 ≈ 3× 順伝播（backward≈2×forward）。粗い近似 $C\approx 6ND$（N=params, D=tokens）。

In [1]:
import warnings
warnings.filterwarnings("ignore")
import json, math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "plotly_mimetype"
from jp_llm_lab.utils.io import repo_root, load_json, read_jsonl
ROOT = repo_root()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
M2_RUN = ROOT / "experiments/runs/m2_model_m_classical_seed42"

In [2]:
from jp_llm_lab.models.config import ModelConfig
from jp_llm_lab.models.transformer import ClassicalGPT
from jp_llm_lab.models.flops import flops_per_token, training_flops
from jp_llm_lab.visualization.params_viz import param_breakdown_figure

cfg = ModelConfig(vocab_size=8192, d_model=320, n_heads=8, n_layers=6, context_len=512)
model = ClassicalGPT(cfg)
bd = model.param_breakdown()
param_breakdown_figure(bd).show()

In [3]:
fp = flops_per_token(cfg, T=512)
print("forward FLOPs/token @T=512:", f'{fp["forward_per_token"]:,}')
print("  per-layer breakdown:", {k: f'{v:,}' for k,v in fp["per_layer"].items()})
print("  lm_head:", f'{fp["lm_head"]:,}')

tokens = 9_830_400  # Model M actual training tokens
tf = training_flops(cfg, tokens, n_params=bd["total"])
print(f"\ntraining FLOPs over {tokens:,} tokens:")
print(f'  exact-ish  : {tf["exact_estimate"]:.3e}')
print(f'  6·N·D approx: {tf["six_ND_approx"]:.3e}')
print(f'  ratio      : {tf["exact_estimate"]/tf["six_ND_approx"]:.2f}')

forward FLOPs/token @T=512: 23,920,640
  per-layer breakdown: {'qkv': '614,400', 'attn_scores(T)': '327,680', 'attn_av(T)': '327,680', 'attn_proj': '204,800', 'mlp': '1,638,400'}
  lm_head: 5,242,880

training FLOPs over 9,830,400 tokens:
  exact-ish  : 7.054e+14
  6·N·D approx: 6.007e+14
  ratio      : 1.17


## Observation / Interpretation / Caveat
- **Observation**: MLP と token embedding が支配的。厳密見積りと 6ND 近似は同オーダーだが一致はしない（lm_head と attention の T 項が近似では簡略化されるため）。
- **Interpretation**: 6ND は便利な目安だが、小語彙・短文脈では lm_head/attention の寄与で数十%ずれる。スケーリング則の議論では厳密側を使うべき場面がある。
- **Caveat**: FLOPs はハードウェアの実効効率（NB01の実測 TFLOPs）とは別物。実時間は kernel 効率・メモリ帯域に左右される。

**次へ**: [13_training_dynamics](13_training_dynamics.ipynb)